# 01.5 V2 — CC Candidate URL Review

Review the Common Crawl URL candidates collected by `scripts/crawl_query.py`.
Check type distribution, TLD breakdown, and URL quality before WDC download.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse
from io import BytesIO
from IPython.display import Image as IPImage

CC_PATH = Path('../data/raw/cc_candidate_urls.jsonl')

records = []
if CC_PATH.exists():
    with open(CC_PATH) as f:
        for line in f:
            records.append(json.loads(line))
    print(f'Loaded {len(records):,} CC candidate URLs')
else:
    print(f'Not found: {CC_PATH}  — run scripts/crawl_query.py first')

In [ ]:
# --- Type distribution ---
type_counts = Counter(r['probable_type'] for r in records)

# --- TLD breakdown ---
def get_tld(url):
    host = urlparse(url).netloc.lower().lstrip('www.')
    parts = host.split('.')
    # Handle co.uk, com.au etc.
    if len(parts) >= 3 and parts[-2] in ('co', 'com', 'org', 'net'):
        return '.'.join(parts[-2:])
    return parts[-1] if parts else 'unknown'

tld_counts = Counter(get_tld(r['url']) for r in records)

# --- URL path sample ---
print('TYPE DISTRIBUTION')
print('-' * 40)
for t, n in type_counts.most_common():
    bar = '█' * int(n / max(type_counts.values()) * 30)
    print(f'  {t:20s} {n:6,}  {bar}')

print()
print('TLD BREAKDOWN')
print('-' * 40)
for tld, n in tld_counts.most_common(10):
    print(f'  .{tld:15s} {n:6,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CC Candidate URLs — 01_v2 Output', fontsize=13, fontweight='bold')

# Type distribution
ax = axes[0]
types, tcounts = zip(*type_counts.most_common())
colours = plt.cm.tab10.colors[:len(types)]
ax.barh(range(len(types)), tcounts, color=colours)
ax.set_yticks(range(len(types)))
ax.set_yticklabels(types)
ax.set_xlabel('URL count')
ax.set_title('@type distribution')
ax.invert_yaxis()
for i, v in enumerate(tcounts):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=8)

# TLD pie
ax = axes[1]
top_tlds = tld_counts.most_common(6)
tld_labels, tld_vals = zip(*top_tlds)
ax.pie(tld_vals, labels=[f'.{t}' for t in tld_labels], autopct='%1.0f%%', startangle=90)
ax.set_title('TLD breakdown')

plt.tight_layout()
buf = BytesIO()
fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
plt.close()
buf.seek(0)
display(IPImage(data=buf.getvalue()))

In [ ]:
# Sample URLs per type — eyeball quality
import random
by_type = {}
for r in records:
    by_type.setdefault(r['probable_type'], []).append(r['url'])

print('SAMPLE URLs per type (5 each)')
print('=' * 60)
for t, urls in sorted(by_type.items()):
    print(f'\n{t}:')
    for url in random.sample(urls, min(5, len(urls))):
        print(f'  {url}')